In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict ,Literal
from pydantic import BaseModel , Field
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
model = ChatGroq(
    model="groq/compound-mini",
    temperature=0.7,
    max_tokens=1000
    )

In [6]:
# Example: e-commerce order processing workflow implemented as a state-graph.
# Each node is a small pure function that receives the current state and returns
# a dict with keys to merge into the state. The StateGraph runs nodes in order
# and can branch conditionally using functions that return the next node name.
#
# Key ideas:
# - State: a dictionary (typed here with TypedDict) that carries data between nodes.
# - Node: a function that accepts state and returns a partial state dict (updates).
# - START / END: special graph anchors for beginning and end of the workflow.
# - add_conditional_edges(node_name, cond_fn): cond_fn examines state and returns
#   the name of the node to execute next (i.e., branching).
# - compile(): prepares/validates the graph and returns a callable workflow.
# - invoke(initial_state): executes the compiled workflow starting from START.
#
# Notes:
# - Nodes should avoid side effects where possible. For IO / external calls,
#   return identifiers/flags in the state and handle retries/async outside.
# - The compiled workflow will merge returned dicts into the shared state.
# - Validation: real systems should validate and sanitize all inputs (not fully shown).

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, List, Dict

# ---- Type definitions (schemas) ----
# TypedDict describes the expected shape of the state dictionary.
# This provides helpful static typing and documentation for each key.
class Item(TypedDict):
    sku: str
    qty: int
    price: float

class OrderState(TypedDict):
    # Basic order identifiers
    order_id: str
    customer_id: str

    # Items list: the order payload
    items: List[Item]

    # Calculated numeric fields (these will be added by nodes)
    subtotal: float
    shipping: float
    tax: float
    total: float

    # Flags and statuses used for branching decisions
    inventory_ok: bool
    payment_status: str  # "success" | "failed" | "pending"

    # Human readable result/log of what happened
    result: str

# ---- Node functions ----
# Each node accepts the current state and returns a dict with keys to merge
# into the state. If a node returns an empty dict, it means "no change".
# Returned values are shallow-merged into the current state.

def show_order(state: OrderState):
    # Purpose: normalization, logging or preparing the order for downstream nodes
    # Example: convert ints/floats, normalize keys, log to audit trail.
    # Here we do nothing and return an empty dict (no state changes).
    return {}

def validate_order(state: OrderState):
    # Purpose: fail-fast validation of incoming order data.
    # If validation fails, we can return a result that indicates an error and
    # the workflow can route or end accordingly (in this simple graph we
    # simply set result; in production you'd branch to an "invalid_order" node).
    if not state.get("items"):
        # Returning result sets state['result'] and the workflow could stop or
        # let subsequent nodes detect this and end early.
        return {"result": "invalid_order_no_items"}
    for it in state["items"]:
        # Basic item-level checks
        if it["qty"] <= 0 or it["price"] < 0:
            return {"result": "invalid_item_quantity_or_price"}
    # No changes if validation passed
    return {}

def calculate_totals(state: OrderState):
    # Purpose: compute subtotal, shipping, tax, and total.
    # This node reads items and returns computed fields that later nodes use.
    subtotal = sum(it["qty"] * it["price"] for it in state["items"])
    # Simple shipping rule: free shipping for orders >= $50
    shipping = 5.0 if subtotal < 50 else 0.0
    # Simple flat tax rate (8%) for demo purposes
    tax = round(subtotal * 0.08, 2)
    total = round(subtotal + shipping + tax, 2)
    # Returning these keys merges them into the shared state
    return {"subtotal": subtotal, "shipping": shipping, "tax": tax, "total": total}

def check_inventory(state: OrderState) -> Literal["process_payment", "notify_backorder"]:
    # Purpose: inspect items and decide which branch to take next.
    # Important: this is a conditional function (it returns a string naming the next node).
    # For demonstration we consider any line with qty > 10 as "out of stock".
    for it in state["items"]:
        if it["qty"] > 10:
            # return the node name that the graph should follow next
            return "notify_backorder"
    return "process_payment"

def process_payment(state: OrderState) -> Dict:
    # Purpose: call payment gateway (simulated here).
    # In real code you'd call an external service and handle network/timeout errors.
    # Node returns updates: payment_status and a result message.
    # This function returns a dict (not a conditional branch); branching on its
    # outcome is done by a separate conditional function.
    if state["total"] < 1000:
        # Simulate success for totals below 1000
        return {"payment_status": "success", "result": f"payment_succeeded_{state['order_id']}"}
    else:
        # Simulate failure for large totals (demo only)
        return {"payment_status": "failed", "result": "payment_failed"}

def payment_decision(state: OrderState) -> Literal["fulfill_order", "handle_payment_failed"]:
    # Purpose: inspect payment_status and return the name of the next node (branch).
    # If payment succeeded -> fulfill; otherwise -> handle failure.
    return "fulfill_order" if state.get("payment_status") == "success" else "handle_payment_failed"

def fulfill_order(state: OrderState):
    # Purpose: create shipment / notify warehouse.
    # Here we just return a result message. In practice, you'd enqueue a job or
    # call the fulfillment service and store a shipment id in the state.
    return {"result": f"order_fulfilled_{state['order_id']}"}

def notify_backorder(state: OrderState):
    # Purpose: inform customer and/or schedule backorder processing.
    return {"result": "notified_backorder"}

def handle_payment_failed(state: OrderState):
    # Purpose: log the failure, notify customer, offer retry or alternative payment.
    # In more sophisticated flows this node might set retry counters or enqueue a retry.
    return {"result": "payment_failed_notify_customer"}

# ---- Build the StateGraph ----
# The StateGraph arranges named nodes and edges (including conditional edges).
graph = StateGraph(OrderState)

# Register nodes with the graph. The first argument is the label used in edges
# and in conditional-return values; the second is the function to execute.
graph.add_node("show_order", show_order)
graph.add_node("validate_order", validate_order)
graph.add_node("calculate_totals", calculate_totals)

# For nodes whose sole purpose is to serve as a branching point we often register
# a "no-op" function because the conditional function is separate (check_inventory).
# The node function is executed (here a lambda returning {}), and the graph then
# asks the conditional function which path to take next.
graph.add_node("check_inventory", lambda s: {})  # placeholder node for conditional branching

graph.add_node("process_payment", process_payment)
graph.add_node("payment_decision", lambda s: {})  # placeholder for second conditional

graph.add_node("fulfill_order", fulfill_order)
graph.add_node("notify_backorder", notify_backorder)
graph.add_node("handle_payment_failed", handle_payment_failed)

# ---- Connect nodes with edges ----
# START and END are special constants used to anchor the workflow.
graph.add_edge(START, "show_order")
graph.add_edge("show_order", "validate_order")
graph.add_edge("validate_order", "calculate_totals")
graph.add_edge("calculate_totals", "check_inventory")

# Conditional branching: when the node "check_inventory" finishes, the graph will
# call the provided function (check_inventory) to decide which named node to run next.
# The function must return one of the node names previously added to the graph.
graph.add_conditional_edges("check_inventory", check_inventory)

# After process_payment node completes, we route to "payment_decision" node which
# is a placeholder; then the conditional function payment_decision decides the next node.
graph.add_edge("process_payment", "payment_decision")
graph.add_conditional_edges("payment_decision", payment_decision)

# Terminal edges: these nodes lead to the END of the graph.
graph.add_edge("fulfill_order", END)
graph.add_edge("notify_backorder", END)
graph.add_edge("handle_payment_failed", END)

# Compile the graph into an executable workflow object.
# compile() typically validates the configuration (nodes/edges) and prepares any
# runtime structures needed for efficient execution.
workflow = graph.compile()

# ---- Example run ----
# Provide an initial state that has the minimal required keys (order_id, customer_id, items).
# Note: the typed keys like subtotal/shipping/tax will be added by nodes during execution.
initial_state = {
    "order_id": "ORD-001",
    "customer_id": "CUST-123",
    "items": [{"sku": "SKU1", "qty": 2, "price": 19.99}],
}

# invoke() runs the compiled workflow from START to END.
# It returns the final state (the same object may be mutated/merged during execution).
result_state = workflow.invoke(initial_state)

# The printed result shows all keys present in the final state after node updates.
print(result_state)

{'order_id': 'ORD-001', 'customer_id': 'CUST-123', 'items': [{'sku': 'SKU1', 'qty': 2, 'price': 19.99}], 'subtotal': 39.98, 'shipping': 5.0, 'tax': 3.2, 'total': 48.18, 'payment_status': 'success', 'result': 'order_fulfilled_ORD-001'}
